# Update overview — derniers matchs téléchargés + état global

Ce notebook sert à :
- inspecter le **dernier run** `data/raw/updates/run_.../` (fichiers, stats avant/après, erreurs)
- voir les **matchs/séries du run** (noms d'équipes + scores finaux)
- voir un **overview global** de `data/processed/` (volume, plage de dates, tendances récentes)


In [1]:
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists() and (p / "data").exists()), None)
if ROOT is None:
    raise RuntimeError("Project root not found (missing 'src' and 'data' folders in parents of cwd).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/home/ju/Documents/Dev/MachineLearning/Dota-Datas')

In [2]:
import json
from datetime import datetime, timezone

import polars as pl
import pandas as pd

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)

PROCESSED_DIR = ROOT / "data/processed"
UPDATES_DIR = ROOT / "data/raw/updates"
TEAMS_CSV = ROOT / "data/teams_to_look.csv"

PROCESSED_DIR, UPDATES_DIR

(PosixPath('/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/processed'),
 PosixPath('/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/raw/updates'))

## 1) Sélection du run

Par défaut, on prend le dernier dossier `run_*` qui contient `matches_trace.csv` (sinon le dernier par mtime).

In [3]:
def _latest_run_dir(updates_dir: Path) -> Path:
    runs = sorted([p for p in updates_dir.glob("run_*") if p.is_dir()])
    if not runs:
        raise FileNotFoundError(f"No run_* directories found in {updates_dir}")
    def _read_json(p: Path):
        try:
            return json.loads(p.read_text()) if p.exists() else None
        except Exception:
            return None

    def _run_score(run_dir: Path) -> tuple[int, float]:
        # Prefer runs that actually downloaded match details.
        score = 0
        run_result = _read_json(run_dir / "run_result.json") or {}
        dl = run_result.get("download_summary") or {}
        if isinstance(dl, dict) and isinstance(dl.get("total_ids"), int):
            score = int(dl.get("total_ids"))
        if score == 0:
            summary = _read_json(run_dir / "summary.json") or {}
            if isinstance(summary, dict) and isinstance(summary.get("total_ids"), int):
                score = int(summary.get("total_ids"))
        if score == 0:
            # fallback: number of chunk files
            score = len(list(run_dir.glob("matches_chunk_*.json")))
        return score, run_dir.stat().st_mtime

    # Pick highest downloaded count; break ties by mtime.
    return max(runs, key=_run_score)

RUN_DIR = _latest_run_dir(UPDATES_DIR)
RUN_DIR

PosixPath('/home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/raw/updates/run_20260123_162225')

## 2) Fichiers de run (metadata / résultat / erreurs)

Les fichiers importants à la fin d'un `make update` :
- `run_metadata.json` (détection, fenêtre temporelle, match_ids)
- `run_result.json` (timings + résumé download)
- `dataset_stats_before/after/delta.json` (avant/après append)
- `matches_trace.csv` et `series_trace.csv` (logs lisibles)


In [4]:
def _read_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())

run_metadata = _read_json(RUN_DIR / "run_metadata.json") or {}
run_result = _read_json(RUN_DIR / "run_result.json") or {}
delta = _read_json(RUN_DIR / "dataset_stats_delta.json") or {}

display(pd.json_normalize(run_metadata))
display(pd.json_normalize(run_result))
display(pd.json_normalize(delta))

,run_id,teams_csv,processed_dir,raw_updates_dir,limit,max_pages,since,chunk_size,sleep_match_detail,sleep_pages,...,per_team_new_counts.8574561,per_team_new_counts.9359300,per_team_new_counts.9546950,per_team_new_counts.8591706,per_team_new_counts.9646467,per_team_new_counts.9264398,per_team_new_counts.9471507,per_team_new_counts.9886133,per_team_new_counts.9730040,per_team_new_counts.8944230
0,20260123_162225,data/teams_to_look.csv,data/processed,data/raw/updates,100,3,None,100,1.0,0.0,...,0,0,0,0,0,0,0,0,0,30


,run_id,elapsed_seconds,discover_seconds,download_seconds,download_summary.total_ids,download_summary.chunk_size,download_summary.resume,download_summary.chunks_written,download_summary.skipped_chunks,download_summary.errors_initial,download_summary.errors_remaining,download_summary.retry_saved,download_summary.out_dir,download_summary.prefix
0,20260123_162225,3483.366108,81.536045,3401.82583,1922,100,False,20,0,0,0,0,data/raw/updates/run_20260123_162225,matches_chunk


,matches_unique_delta,series_rows_delta,series_pairs_unique_delta
0,1922,1007,1007


In [5]:
files = sorted([p.name for p in RUN_DIR.iterdir() if p.is_file()])
pl.DataFrame({"files": files})

files
str
"""dataset_stats_after.json"""
"""dataset_stats_before.json"""
"""dataset_stats_delta.json"""
"""healthcheck.json"""
"""matches_chunk_0000.json"""
"""matches_chunk_0001.json"""
"""matches_chunk_0002.json"""
"""matches_chunk_0003.json"""
"""matches_chunk_0004.json"""


## 3) Matchs et séries du run (logs lisibles)

Ces CSV sont faits pour être *lisibles* rapidement (noms d'équipes, résultat final de série, date).

In [6]:
matches_trace_path = RUN_DIR / "matches_trace.csv"
series_trace_path = RUN_DIR / "series_trace.csv"
matches_trace_parquet = RUN_DIR / "matches_trace.parquet"
series_trace_parquet = RUN_DIR / "series_trace.parquet"

matches_trace = pl.read_csv(matches_trace_path, raise_if_empty=False) if matches_trace_path.exists() else pl.DataFrame([])
if matches_trace.is_empty() and matches_trace_parquet.exists():
    matches_trace = pl.read_parquet(matches_trace_parquet)

series_trace = pl.read_csv(series_trace_path, raise_if_empty=False) if series_trace_path.exists() else pl.DataFrame([])
if series_trace.is_empty() and series_trace_parquet.exists():
    series_trace = pl.read_parquet(series_trace_parquet)

print("matches_trace rows:", matches_trace.height)
print("series_trace rows:", series_trace.height)

series_trace.head(20)

matches_trace rows: 1922
series_trace rows: 1008


start_dt_run,tournament_name,team_a_name,team_b_name,score_team_a,score_team_b,winner_team_name,series_id,leagueid
str,str,str,str,str,str,str,i64,i64
"""2026-01-23T12:30:09.000000+000…",null,null,null,null,null,null,1057615,19239
"""2026-01-20T16:02:44.000000+000…",null,null,null,null,null,null,1056845,18941
"""2026-01-20T15:03:59.000000+000…",null,null,null,null,null,null,1056834,19090
"""2026-01-20T13:25:03.000000+000…",null,null,null,null,null,null,1056809,18941
"""2026-01-20T11:02:05.000000+000…",null,null,null,null,null,null,1056788,19090
"""2026-01-20T10:07:24.000000+000…",null,null,null,null,null,null,1056782,18941
"""2026-01-19T21:18:00.000000+000…",null,null,null,null,null,null,1056689,19090
"""2026-01-19T19:00:32.000000+000…",null,null,null,null,null,null,1056654,19090
"""2026-01-19T19:45:13.000000+000…",null,null,null,null,null,null,1056671,18941


In [7]:
if not series_trace.is_empty() and "tournament_name" in series_trace.columns:
    top = (
        series_trace
        .group_by("tournament_name")
        .agg(pl.len().alias("series"))
        .sort("series", descending=True)
    )
    top.head(25)
else:
    print("series_trace.csv missing or empty")

In [8]:
matches_trace.head(30)

start_dt,match_id,radiant_name,dire_name,radiant_win,leagueid,series_id,map_num
str,i64,str,str,bool,i64,i64,str
"""2026-01-23T13:45:06.000000+000…",8660862355,"""AVULUS""","""1w Team""",false,19239,1057615,null
"""2026-01-23T12:30:09.000000+000…",8660759083,"""1w Team""","""AVULUS""",true,19239,1057615,null
"""2026-01-20T18:59:14.000000+000…",8657549796,"""MONKEY BUSINESS""","""Zero Tenacity""",false,18941,1056845,null
"""2026-01-20T17:57:23.000000+000…",8657479892,"""MONKEY BUSINESS""","""Zero Tenacity""",true,18941,1056845,null
"""2026-01-20T17:47:13.000000+000…",8657450979,"""Nigma Galaxy""","""Natus Vincere""",true,19090,1056834,null
"""2026-01-20T16:28:13.000000+000…",8657369967,"""Nigma Galaxy""","""Natus Vincere""",true,19090,1056834,null
"""2026-01-20T16:02:44.000000+000…",8657321801,"""Zero Tenacity""","""MONKEY BUSINESS""",true,18941,1056845,null
"""2026-01-20T15:03:59.000000+000…",8657246305,"""Nigma Galaxy""","""Natus Vincere""",false,19090,1056834,null
"""2026-01-20T14:27:41.000000+000…",8657208556,"""Team Spirit Academy""","""Rune Eaters""",false,18941,1056809,null


## 4) Overview global (data/processed)

On se limite volontairement à `matches.parquet`, `series.parquet`, `extras.parquet` (pas de lecture de `players.parquet`).

In [9]:
matches_path = PROCESSED_DIR / "matches.parquet"
series_path = PROCESSED_DIR / "series.parquet"
extras_path = PROCESSED_DIR / "extras.parquet"

lf = pl.scan_parquet(matches_path)
cols = lf.collect_schema().names()

exprs = []
if "match_id" in cols:
    exprs += [pl.col("match_id").n_unique().alias("matches_unique"), pl.len().alias("matches_rows")]
if "start_time" in cols:
    exprs += [pl.col("start_time").min().alias("start_time_min"), pl.col("start_time").max().alias("start_time_max")]
if "series_id" in cols and "leagueid" in cols:
    exprs += [
        pl.struct(["series_id", "leagueid"]).n_unique().alias("series_pairs_unique")
    ]

global_stats = lf.select(exprs).collect().to_dicts()[0] if exprs else {}

def _ts_to_dt(ts):
    if ts is None:
        return None
    return datetime.fromtimestamp(int(ts), tz=timezone.utc)

global_stats["start_dt_min"] = _ts_to_dt(global_stats.get("start_time_min"))
global_stats["start_dt_max"] = _ts_to_dt(global_stats.get("start_time_max"))

series_rows = pl.scan_parquet(series_path).select(pl.len().alias("series_rows")).collect().item() if series_path.exists() else None
extras_unique = pl.scan_parquet(extras_path).select(pl.col("match_id").n_unique()).collect().item() if extras_path.exists() else None

global_stats["series_rows"] = int(series_rows) if series_rows is not None else None
global_stats["extras_unique"] = int(extras_unique) if extras_unique is not None else None

pd.DataFrame([global_stats])

,matches_unique,matches_rows,start_time_min,start_time_max,series_pairs_unique,start_dt_min,start_dt_max,series_rows,extras_unique
0,14913,14913,1704355281,1769175906,6953,2024-01-04 08:01:21+00:00,2026-01-23 13:45:06+00:00,6952,743


In [10]:
teams_df = pl.read_csv(TEAMS_CSV)
teams_df = teams_df.rename({c: c.strip() for c in teams_df.columns})
tracked_ids = [int(x) for x in teams_df["TeamID"].to_list()]

needed = [c for c in ["match_id", "start_time", "radiant_team_id", "dire_team_id", "tournament_name", "league_name"] if c in cols]
m = lf.select(needed).collect()

tracked_matches = m.filter((pl.col("radiant_team_id").is_in(tracked_ids)) | (pl.col("dire_team_id").is_in(tracked_ids))) if {"radiant_team_id","dire_team_id"}.issubset(set(m.columns)) else pl.DataFrame([])

out = {
    "tracked_teams": len(tracked_ids),
    "tracked_matches_unique": int(tracked_matches.select(pl.col("match_id").n_unique()).item()) if not tracked_matches.is_empty() else 0,
}
pd.DataFrame([out])

,tracked_teams,tracked_matches_unique
0,174,14913


## 5) Fenêtre récente (derniers matchs globalement)

Affiche les derniers matchs de `matches.parquet` (utile pour vérifier rapidement que l'update fait avancer la date).

In [11]:
recent_cols = [c for c in ["start_time", "match_id", "radiant_name", "dire_name", "radiant_win", "league_name", "tournament_name", "series_id", "map_num"]]
recent_cols = [c for c in recent_cols if c in cols]

recent = (
    pl.scan_parquet(matches_path)
    .select(recent_cols)
    .sort("start_time", descending=True)
    .head(50)
    .collect()
)
if "start_time" in recent.columns:
    recent = recent.with_columns(pl.from_epoch(pl.col("start_time"), time_unit="s").dt.replace_time_zone("UTC").alias("start_dt"))
recent.select([c for c in ["start_dt", *recent_cols] if c in recent.columns]).head(30)

start_dt,start_time,match_id,radiant_name,dire_name,radiant_win,league_name,tournament_name,series_id
"datetime[μs, UTC]",i64,i64,str,str,bool,str,str,i64
2026-01-23 13:45:06 UTC,1769175906,8660862355,"""AVULUS""","""1w Team""",false,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057615
2026-01-23 12:30:09 UTC,1769171409,8660759083,"""1w Team""","""AVULUS""",true,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057615
2026-01-23 10:03:01 UTC,1769162581,8660611286,"""Pipsqueak+4""","""L1GA TEAM""",true,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057591
2026-01-23 09:00:09 UTC,1769158809,8660564089,"""L1GA TEAM""","""Pipsqueak+4""",false,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057591
2026-01-22 21:01:00 UTC,1769115660,8660138752,"""L1GA TEAM""","""1000 reasons""",true,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057391
2026-01-22 19:36:09 UTC,1769110569,8660048910,"""L1GA TEAM""","""1000 reasons""",true,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057391
2026-01-22 17:25:06 UTC,1769102706,8659901193,"""Nigma Galaxy""","""Pipsqueak+4""",true,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057336
2026-01-22 16:16:06 UTC,1769098566,8659815119,"""Pipsqueak+4""","""Nigma Galaxy""",false,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057336
2026-01-22 14:44:30 UTC,1769093070,8659687753,"""AVULUS""","""Inner Circle x Insanity""",false,"""FISSURE Universe Episode 8""","""FISSURE Universe Episode 8""",1057304
